# 1주차

KNN

`CSV 로드 -> 타겟 생성 -> 타입 정리 -> Train/Val 분리 -> 누수 방지 전처리 -> SMOTE -> kNN 튜닝 -> PCA 시각화 -> Test 예측 -> CSV 저장 -> NumPy로 kNN 재구현`


## 1. 라이브러리 프레임워크

핵심 역할:

- `pandas`: CSV 로드, 표 데이터 다루기
- `numpy`: 배열 연산, 거리 계산, 브로드캐스팅
- `scikit-learn`: 전처리, 분할, 모델 학습, 평가, PCA
- `imblearn`: SMOTE로 클래스 불균형 보정

원리:

- 전체 파이프라인은 보통 `DataFrame -> numpy array -> model input` 흐름으로 간다.
- 실무에서는 데이터를 읽는 도구와 모델 학습 도구가 분리되어 있기 때문에, 각 라이브러리의 역할을 같이 기억하는 것이 중요하다.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE


## 2. 데이터 로드와 타겟 생성

핵심 문법:

- `pd.read_csv(path)`: CSV를 `DataFrame`으로 읽는다.
- `df['col']`: 특정 열을 선택한다.
- `Series.apply(func)`: 각 원소에 함수를 적용한다.
- `lambda x: ...`: 짧은 익명 함수다.

원리:

- 원래 데이터의 `transmission`은 문자열이다.
- 모델은 문자열 자체를 바로 예측하기보다, `Auto / Manual`을 `0 / 1`로 바꾼 타겟을 학습한다.

흐름:

- 원본 CSV 로드
- `transmission` 열 확인
- 규칙 기반으로 `target` 열 생성


In [ ]:
trainval_df = pd.read_csv('trainval.csv')
test_df = pd.read_csv('test.csv')

trainval_df['target'] = trainval_df['transmission'].apply(
    lambda x: 1 if 'Manual' in str(x) else 0
)


## 3. 타입 정리와 Train/Val 분리

핵심 문법:

- `pd.to_numeric(series, errors='coerce')`
- `astype(str)`
- `train_test_split(X, y, test_size=0.2)`

원리:

- 전처리 전에 먼저 각 열의 타입을 확정해야 한다.
- 수치형은 숫자로 강제 변환하고, 실패한 값은 `NaN`으로 바꿔 나중에 결측치 처리한다.
- 범주형은 문자열로 통일해야 인코더가 안정적으로 동작한다.

중요 포인트:

- 데이터를 먼저 `Train`과 `Val`로 나누는 이유는 **Data Leakage**를 막기 위해서다.
- 이후 기준 학습은 반드시 `Train`으로만 해야 한다.


In [ ]:
numeric_cols = [
    'year',
    'engine_cylinders',
    'engine_displacement',
    'city_mpg_ft1',
    'highway_mpg_ft1',
]

categorical_cols = [
    'make',
    'class',
    'drive',
]

for col in numeric_cols:
    trainval_df[col] = pd.to_numeric(trainval_df[col], errors='coerce')
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

for col in categorical_cols:
    trainval_df[col] = trainval_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

feature_cols = numeric_cols + categorical_cols

X_train_full, X_val_full, y_train, y_val = train_test_split(
    trainval_df[feature_cols],
    trainval_df['target'],
    test_size=0.2,
    random_state=42,
)


## 4. 전처리 파이프라인 문법

핵심 문법:

- `Pipeline(steps=[...])`
- `SimpleImputer(strategy='median')`
- `SimpleImputer(strategy='most_frequent')`
- `StandardScaler()`
- `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`
- `ColumnTransformer(transformers=[...])`

원리:

- 수치형과 범주형은 처리 방식이 다르기 때문에 열 그룹별 체인을 따로 만든다.
- `Pipeline`은 같은 종류의 처리 순서를 묶는 도구다.
- `ColumnTransformer`는 여러 파이프라인을 열 그룹별로 병렬 적용하는 도구다.

흐름:

- 수치형: 결측치 대체 -> 스케일링
- 범주형: 결측치 대체 -> 원-핫 인코딩
- 마지막에 두 결과를 하나의 입력 행렬로 합친다.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)


## 5. `fit`과 `transform`

핵심 문법:

- `preprocessor.fit(X_train_full)`
- `preprocessor.transform(X_val_full)`

원리:

- `fit`: 기준을 학습한다.
- `transform`: 이미 학습한 기준을 적용한다.

왜 중요한가:

- 중앙값, 평균, 분산, 범주 목록 같은 기준을 `Val/Test`까지 보고 만들면 미래 정보를 미리 본 것과 같아진다.
- 그래서 `Train`으로만 `fit`하고, `Val/Test`에는 `transform`만 해야 한다.


In [ ]:
preprocessor.fit(X_train_full)

X_train_scaled = preprocessor.transform(X_train_full)
X_val_scaled = preprocessor.transform(X_val_full)
X_test_scaled = preprocessor.transform(test_df[feature_cols])


## 6. 클래스 불균형 처리: SMOTE

핵심 문법:

- `SMOTE(k_neighbors=2)`
- `fit_resample(X, y)`

원리:

- 소수 클래스 주변의 이웃을 참고해서 새로운 가상 샘플을 만든다.
- 단순 복제가 아니라, 기존 샘플 사이를 보간해서 데이터를 늘린다.

중요 포인트:

- SMOTE는 반드시 `Train`에만 적용한다.
- `Val/Test`에 적용하면 현실 분포가 깨져서 평가가 왜곡된다.


In [ ]:
smote = SMOTE(k_neighbors=2)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)


## 7. kNN 학습과 튜닝

핵심 문법:

- `KNeighborsClassifier(n_neighbors=k)`
- `.fit(X, y)`
- `.predict(X)`
- `accuracy_score(y_true, y_pred)`
- `f1_score(..., average='macro')`

원리:

- kNN은 가장 가까운 이웃 `k`개의 라벨을 보고 다수결로 분류한다.
- `k`가 너무 작으면 노이즈에 민감하고, 너무 크면 경계가 둔해질 수 있다.

평가 포인트:

- 클래스 불균형이 있으면 `accuracy`만 보면 착시가 생길 수 있다.
- 그래서 각 클래스를 균형 있게 보는 `macro F1`이 더 중요하다.


In [ ]:
best_k = 1
best_score = 0

for k in [1, 3, 5, 7, 9]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_resampled, y_train_resampled)
    val_preds = knn.predict(X_val_scaled)

    acc = accuracy_score(y_val, val_preds)
    f1 = f1_score(y_val, val_preds, average='macro')

    if f1 > best_score:
        best_score = f1
        best_k = k


## 8. 분류 성적표와 혼동행렬

핵심 문법:

- `classification_report(...)`
- `ConfusionMatrixDisplay.from_predictions(...)`

원리:

- 단순 accuracy는 전체 정답률만 보여준다.
- 하지만 혼동행렬은 어떤 클래스를 어떤 클래스로 헷갈렸는지 보여준다.
- 즉, 모델의 약점을 분석하는 도구라고 보면 된다.


In [ ]:
final_knn_model = KNeighborsClassifier(n_neighbors=best_k)
final_knn_model.fit(X_train_resampled, y_train_resampled)

best_val_preds = final_knn_model.predict(X_val_scaled)

print(classification_report(y_val, best_val_preds, target_names=['Auto (0)', 'Manual (1)']))
ConfusionMatrixDisplay.from_predictions(y_val, best_val_preds)


## 9. PCA와 Decision Boundary

핵심 문법:

- `PCA(n_components=2)`
- `fit_transform(X)`
- `np.meshgrid(...)`
- `np.c_[a, b]`
- `ravel()`
- `inverse_transform(X_2d)`
- `reshape(shape)`

원리:

- 고차원 데이터를 PCA로 2차원에 투영해 시각화한다.
- 2차원 평면 위에 격자를 만든 뒤, 그 점들을 다시 원래 차원으로 복원해서 모델에 넣는다.
- 이렇게 하면 실제 고차원 모델의 결정 경계를 2차원 단면처럼 볼 수 있다.


In [ ]:
pca_model = PCA(n_components=2)
X_2d = pca_model.fit_transform(X_train_resampled)

xx, yy = np.meshgrid(
    np.arange(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 0.05),
    np.arange(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 0.05),
)
mesh_2d = np.c_[xx.ravel(), yy.ravel()]
mesh_high_dim = pca_model.inverse_transform(mesh_2d)
Z = final_knn_model.predict(mesh_high_dim).reshape(xx.shape)


## 10. 최종 제출 파일 생성

핵심 문법:

- `.copy()`
- `pd.DataFrame({...})`
- `.to_csv(filename, index=False)`

원리:

- 검증 단계에서 최적 `k`를 찾은 뒤, 과거 데이터 전체(`trainval`)로 다시 학습한다.
- 그다음 미래 데이터(`test`)를 예측해서 제출 파일을 만든다.

흐름:

- `Train/Val` 분리는 튜닝용
- 최종 제출은 `trainval` 전체 재학습


In [ ]:
X_full = trainval_df[feature_cols].copy()
y_full = trainval_df['target']

preprocessor.fit(X_full)
X_full_scaled = preprocessor.transform(X_full)
X_test_scaled = preprocessor.transform(test_df[feature_cols])

X_full_resampled, y_full_resampled = smote.fit_resample(X_full_scaled, y_full)

final_knn_model = KNeighborsClassifier(n_neighbors=best_k)
final_knn_model.fit(X_full_resampled, y_full_resampled)

test_preds = final_knn_model.predict(X_test_scaled)
submission_df = pd.DataFrame({'vehicle_id': test_df['vehicle_id'], 'target': test_preds})
submission_df.to_csv('submission_홍길동.csv', index=False)


## 11. NumPy로 kNN 직접 구현

이 부분은 함수 문법과 클래스 흐름을 같이 보는 것이 중요하다.

### `__init__`

- 객체가 생성될 때 자동 호출되는 초기화 메서드다.
- 여기서 `pass`를 쓴 것은 아직 초기화할 내용을 따로 넣지 않았기 때문이다.
- 즉, 생성자는 열어 두었지만 지금은 아무 동작도 하지 않는 상태다.

### `fit`

- kNN은 가중치를 학습하는 모델이 아니라, 훈련 데이터를 저장해 두는 모델이다.
- 그래서 `fit`의 핵심은 `self.X_train`, `self.y_train`에 데이터를 저장하는 것이다.

### `compute_distances`

핵심 식:

`||a-b||^2 = ||a||^2 - 2a·b + ||b||^2`

- 이 식을 쓰면 이중 반복문 없이 테스트 샘플과 학습 샘플 사이의 거리를 한 번에 계산할 수 있다.
- `axis=1`, `keepdims=True`, 전치(`.T`)가 왜 필요한지 같이 보는 것이 중요하다.

### `predict`

- 거리 행렬에서 가까운 `k`개의 인덱스를 찾는다.
- 그 인덱스의 라벨을 모아 `np.bincount()`로 개수를 세고 `np.argmax()`로 다수 클래스를 고른다.


In [ ]:
class KNearestNeighbor_Numpy:
    def __init__(self):
        pass

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def compute_distances(self, X_test):
        X_test = np.array(X_test)

        test_sq = np.sum(np.square(X_test), axis=1, keepdims=True)
        train_sq = np.sum(np.square(self.X_train), axis=1)
        cross_term = np.dot(X_test, self.X_train.T)

        dists = np.sqrt(np.maximum(test_sq - 2 * cross_term + train_sq, 0.0))
        return dists

    def predict(self, X_test, k=1):
        dists = self.compute_distances(X_test)
        num_test = dists.shape[0]
        y_pred = np.zeros(num_test)

        for i in range(num_test):
            closest_idx = np.argsort(dists[i, :])[:k]
            closest_y = self.y_train[closest_idx]
            counts = np.bincount(closest_y.astype(int))
            y_pred[i] = np.argmax(counts)

        return y_pred


## 12. 외우면 좋은 핵심 문법

```python
df['new_col'] = df['old_col'].apply(lambda x: ...)
pd.to_numeric(series, errors='coerce')
series.astype(str)
train_test_split(X, y, test_size=0.2)
Pipeline(steps=[...])
ColumnTransformer(transformers=[...])
preprocessor.fit(X_train)
preprocessor.transform(X_val)
smote.fit_resample(X, y)
model.fit(X, y)
model.predict(X)
f1_score(y_true, y_pred, average='macro')
PCA(n_components=2)
pca.fit_transform(X)
pca.inverse_transform(X_2d)
np.meshgrid(...)
np.c_[a, b]
np.argsort(arr)[:k]
np.bincount(labels.astype(int))
submission_df.to_csv('file.csv', index=False)
```


# 2주차

LiDAR Point Cloud 분석

`환경 준비 -> 데이터 경로 확인 -> nuScenes 파싱 -> Point Cloud/BBox 시각화 -> 클러스터링 -> bbox feature 추출 -> SVM 학습/평가 -> PCA 결정 경계 -> Scene 2 추론`


## 1. 라이브러리와 환경 준비

핵심 역할:

- `plotly`: 3D point cloud와 bounding box 시각화
- `ipywidgets`: 프레임 슬라이더와 재생 UI
- `PIL`: 카메라 이미지 로드
- `importlib.util.find_spec(...)`: 필수 패키지 설치 여부 확인

원리:

- LiDAR 실습은 단순 계산만이 아니라, 인터랙티브 시각화 환경이 같이 준비되어야 한다.
- 그래서 노트북 시작 단계에서 패키지 체크와 Plotly renderer 설정을 먼저 수행한다.


In [ ]:
import importlib.util
import plotly.io as pio

required_modules = {
    'ipywidgets': 'ipywidgets',
    'plotly': 'plotly',
    'PIL': 'pillow',
}
missing = [pkg_name for module_name, pkg_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]

if missing:
    raise ModuleNotFoundError(
        "필수 패키지가 없습니다: " + ", ".join(missing)
    )

preferred_renderers = [
    'plotly_mimetype+notebook_connected',
    'plotly_mimetype',
    'jupyterlab',
    'notebook_connected',
    'browser',
]

for renderer in preferred_renderers:
    try:
        pio.renderers.default = renderer
        break
    except ValueError:
        continue

print(f"Plotly renderer: {pio.renderers.default}")


## 2. 프로젝트 루트와 데이터 경로 찾기

핵심 문법:

- `Path.cwd()`
- `start.parents`
- `(base / "data" / "v1.0-mini-extract").exists()`
- `os.chdir(PROJECT_ROOT)`

원리:

- 실습 노트북은 실행 위치가 달라질 수 있으므로, 현재 폴더 기준으로 상위 디렉터리를 타고 올라가며 데이터 폴더를 찾는다.
- 이렇게 하면 경로 하드코딩 없이 로컬/주피터 환경에서 같은 코드를 재사용할 수 있다.


In [ ]:
import os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / 'data' / 'v1.0-mini-extract').exists():
            return base
    raise FileNotFoundError("data/v1.0-mini-extract 폴더를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"데이터 경로: {PROJECT_ROOT / 'data' / 'v1.0-mini-extract'}")


## 3. nuScenes 유틸리티와 데이터베이스 클래스

핵심 문법:

- `json.load(f)`
- `{r["token"]: r for r in records}`
- `np.linalg.inv(T)`
- quaternion -> rotation matrix 변환

원리:

- nuScenes는 여러 JSON 테이블로 나뉘어 있으므로, token 기반 인덱스를 먼저 만들어야 필요한 레코드를 빠르게 찾을 수 있다.
- LiDAR 좌표계로 박스를 옮기려면 `global -> sensor` 변환 행렬이 필요하다.


In [ ]:
import json
import numpy as np

DATAROOT = os.path.join(os.path.abspath("."), "data")
VERSION = "v1.0-mini-extract"

def quaternion_to_rotation_matrix(q):
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)]
    ])

def make_transform(translation, rotation):
    T = np.eye(4)
    T[:3, :3] = quaternion_to_rotation_matrix(rotation)
    T[:3, 3] = translation
    return T

class NuScenesDB:
    def __init__(self, dataroot=DATAROOT, version=VERSION):
        self.dataroot = dataroot
        self.version = version
        self.cs_idx   = self._index(self._load("calibrated_sensor"))
        self.ep_idx   = self._index(self._load("ego_pose"))
        self.inst_idx = self._index(self._load("instance"))
        self.cat_idx  = self._index(self._load("category"))

    def _load(self, name):
        path = os.path.join(self.dataroot, self.version, f"{name}.json")
        with open(path) as f:
            return json.load(f)

    @staticmethod
    def _index(records):
        return {r["token"]: r for r in records}

    def get_sensor_transform(self, sd_record):
        cs = self.cs_idx[sd_record["calibrated_sensor_token"]]
        ep = self.ep_idx[sd_record["ego_pose_token"]]
        sensor2global = make_transform(ep["translation"], ep["rotation"]) @ make_transform(cs["translation"], cs["rotation"])
        return np.linalg.inv(sensor2global), cs

    def get_category(self, ann):
        return self.cat_idx[self.inst_idx[ann["instance_token"]]["category_token"]]["name"]


## 4. scene -> sample -> sample_data 파싱

핵심 문법:

- `next(...)`
- `dict.setdefault(key, [])`
- `while tok:`

원리:

- nuScenes는 `scene` 안에 여러 `sample`이 연결 리스트처럼 이어진다.
- 각 sample마다 LiDAR와 6개 카메라 keyframe을 모아 두면, 이후 시각화나 학습용 프레임 로딩이 쉬워진다.


In [ ]:
db = NuScenesDB(DATAROOT, VERSION)

with open(os.path.join(DATAROOT, VERSION, "scene.json")) as f:
    scenes = json.load(f)
with open(os.path.join(DATAROOT, VERSION, "sample.json")) as f:
    samples_list = json.load(f)
with open(os.path.join(DATAROOT, VERSION, "sample_data.json")) as f:
    sd_list = json.load(f)
with open(os.path.join(DATAROOT, VERSION, "sample_annotation.json")) as f:
    ann_list = json.load(f)

samples_idx = {s["token"]: s for s in samples_list}
sd_by_sample = {}
ann_by_sample = {}

for sd in sd_list:
    sd_by_sample.setdefault(sd["sample_token"], []).append(sd)

for ann in ann_list:
    ann_by_sample.setdefault(ann["sample_token"], []).append(ann)

CAM_CHANNELS = [
    "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
    "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT",
]

sync_triplets = {}

for scene in scenes:
    frames = []
    tok = scene["first_sample_token"]
    while tok:
        sample = samples_idx.get(tok)
        if sample is None:
            break
        sds = sd_by_sample.get(tok, [])
        lidar_sd = next((sd for sd in sds if "LIDAR_TOP" in sd["filename"] and sd["is_key_frame"]), None)
        cameras = {ch: next((sd for sd in sds if f"/{ch}/" in sd["filename"] and sd["is_key_frame"]), None) for ch in CAM_CHANNELS}
        if lidar_sd is not None:
            frames.append({
                "timestamp": sample["timestamp"],
                "lidar": lidar_sd,
                "cameras": cameras,
                "annotations": ann_by_sample.get(tok, []),
            })
        tok = sample.get("next", "")
    sync_triplets[scene["token"]] = frames

print(f"scene 수: {len(sync_triplets)}")


## 5. Point Cloud와 Bounding Box 시각화 흐름

핵심 문법:

- `np.fromfile(...).reshape(-1, 5)`
- LiDAR는 보통 `(x, y, z, intensity, ring)` 형식으로 읽는다.
- `FigureWidget` + `widgets.IntSlider` 조합으로 프레임 갱신

원리:

- 한 프레임을 읽을 때는 LiDAR 점군과 annotation을 함께 로드해야 한다.
- Bounding Box는 global 좌표에서 sensor 좌표로 변환한 뒤 edge 좌표를 그린다.


In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from PIL import Image

def load_lidar(sd_record):
    path = os.path.join(DATAROOT, sd_record["filename"])
    return np.fromfile(path, dtype=np.float32).reshape(-1, 5)

def load_frame(scene_token, frame_idx):
    frame_data = sync_triplets[scene_token][frame_idx]
    lidar_sd = frame_data["lidar"]
    anns = frame_data["annotations"]
    pc = load_lidar(lidar_sd)[:, :3]
    global2lidar, _ = db.get_sensor_transform(lidar_sd)
    return pc, anns, global2lidar

def pointcloud_filter(pc):
    return pc[pc[:, 2] >= -1.2]

first_scene_token = list(sync_triplets.keys())[0]
pc, anns, global2lidar = load_frame(first_scene_token, 0)
pc = pointcloud_filter(pc)
print(f"point cloud shape: {pc.shape}")
print(f"annotation 수: {len(anns)}")


## 6. K-Means와 DBSCAN 군집화

핵심 문법:

- `KMeans(n_clusters=K)`
- `DBSCAN(eps=..., min_samples=...)`
- `fit_predict(X)`

원리:

- K-Means는 군집 개수를 미리 정해야 하고, 모든 점을 반드시 어떤 군집에 넣는다.
- DBSCAN은 밀도가 낮은 점을 `-1` 노이즈로 분리할 수 있어서 LiDAR 객체 추출에 더 자연스럽다.


In [ ]:
from sklearn.cluster import KMeans, DBSCAN
import copy

pc, _, _ = load_frame(first_scene_token, 0)
X_pc = copy.copy(pointcloud_filter(pc))

K = 17
kmeans = KMeans(n_clusters=K, random_state=42, n_init="auto")
labels_kmeans = kmeans.fit_predict(X_pc)

dbscan = DBSCAN(eps=0.9, min_samples=20)
labels_dbscan = dbscan.fit_predict(X_pc)

n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = list(labels_dbscan).count(-1)

print(f"K-Means 군집 수: {len(set(labels_kmeans))}")
print(f"DBSCAN 군집 수: {n_clusters}, 노이즈 수: {n_noise}")


## 7. GT Bounding Box 내부 포인트 추출과 feature engineering

핵심 문법:

- `(pc_global[:, :3] - center) @ R`
- `np.abs(...) <= size / 2`
- `max() - min()`, `mean()`, `std()`

원리:

- 박스 내부 여부를 판단하려면, 포인트를 박스 로컬 좌표계로 옮긴 뒤 축 정렬 박스 조건으로 검사하면 된다.
- 그 후 포인트 수, 높이 범위, 가로/세로 범위 같은 기하학적 특징을 추출해 분류기 입력으로 사용한다.


In [ ]:
def extract_bbox_points(pc_global, ann):
    R = quaternion_to_rotation_matrix(ann["rotation"])
    center = np.array(ann["translation"])
    w, l, h = ann["size"]

    pts_local = (pc_global[:, :3] - center) @ R

    mask = (
        (np.abs(pts_local[:, 0]) <= l / 2) &
        (np.abs(pts_local[:, 1]) <= w / 2) &
        (np.abs(pts_local[:, 2]) <= h / 2)
    )
    return pc_global[mask, :3]

def compute_features(pts):
    if len(pts) == 0:
        return None
    n_pts = len(pts)
    z_range = pts[:, 2].max() - pts[:, 2].min()
    x_range = pts[:, 0].max() - pts[:, 0].min()
    y_range = pts[:, 1].max() - pts[:, 1].min()
    z_mean = pts[:, 2].mean()
    z_std = pts[:, 2].std()
    return np.array([n_pts, z_range, x_range, y_range, z_mean, z_std])


## 8. Scene 단위 샘플 수집

핵심 문법:

- `np.hstack([pc, np.ones((len(pc), 1))])`
- homogeneous coordinate
- `for frame in sync_triplets[scene_token]`

원리:

- 학습 데이터는 CSV처럼 이미 준비된 것이 아니라, 각 frame의 GT bbox에서 직접 feature를 뽑아 만들어야 한다.
- Scene 1을 train, Scene 2를 test로 두어 일반화 성능을 확인한다.


In [ ]:
LABEL_MAP = {"vehicle": 1, "pedestrian": -1}
MIN_PTS = 5

def collect_samples(scene_token):
    X, y = [], []
    for frame in sync_triplets[scene_token]:
        lidar_sd = frame["lidar"]
        pc_full = load_lidar(lidar_sd)
        global2lidar, _ = db.get_sensor_transform(lidar_sd)
        lidar2global = np.linalg.inv(global2lidar)

        pts_h = np.hstack([pc_full[:, :3], np.ones((len(pc_full), 1))])
        pc_global = (lidar2global @ pts_h.T).T[:, :3]

        for ann in frame["annotations"]:
            cat = db.get_category(ann)
            if cat.startswith("vehicle"):
                label = LABEL_MAP["vehicle"]
            elif cat.startswith("human.pedestrian"):
                label = LABEL_MAP["pedestrian"]
            else:
                continue

            pts_inside = extract_bbox_points(pc_global, ann)
            if len(pts_inside) < MIN_PTS:
                continue

            feat = compute_features(pts_inside)
            if feat is None:
                continue

            X.append(feat)
            y.append(label)

    return np.array(X), np.array(y)

scene_tokens = list(sync_triplets.keys())
TRAIN_SCENE = scene_tokens[0]
TEST_SCENE = scene_tokens[1]

X_train, y_train = collect_samples(TRAIN_SCENE)
X_test, y_test = collect_samples(TEST_SCENE)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


## 9. SVM 학습과 정량 평가

핵심 문법:

- `make_pipeline(StandardScaler(), SVC(...))`
- `accuracy_score`, `balanced_accuracy_score`
- `precision_recall_fscore_support`
- `confusion_matrix`

원리:

- feature마다 단위가 다르므로 SVM 전에 반드시 `StandardScaler`로 정규화한다.
- 자율주행 데이터는 클래스 불균형이 있을 수 있으므로 accuracy만 보지 않고 balanced accuracy와 macro F1도 함께 확인한다.


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix

svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=3.0, gamma="scale", class_weight="balanced")
)
svm.fit(X_train, y_train)

def evaluate_split(name, X, y_true):
    y_pred = svm.predict(X)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[1, -1], zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[1, -1])

    print(f"[{name}] Accuracy: {acc:.1%}")
    print(f"[{name}] Balanced Accuracy: {bal_acc:.1%}")
    print(f"[{name}] Macro F1: {np.mean(f1):.3f}")
    print(cm)
    return y_pred

y_pred_train = evaluate_split("Train", X_train, y_train)
y_pred_test = evaluate_split("Test", X_test, y_test)


## 10. PCA와 Decision Boundary

핵심 문법:

- `PCA(n_components=2)`
- `pca.fit_transform(X_scaled)`
- `pca.inverse_transform(X_2d)`
- `np.meshgrid(...)`

원리:

- 실제 SVM은 6차원 feature 공간에서 동작한다.
- 하지만 사람이 보기에는 어렵기 때문에 PCA로 2차원에 투영해 결정 경계 모양을 직관적으로 확인한다.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
scaler = svm.named_steps["standardscaler"]
X_scaled = scaler.transform(X_train)
X_2d = pca.fit_transform(X_scaled)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

grid_6d = pca.inverse_transform(np.c_[xx.ravel(), yy.ravel()])
Z = svm.named_steps["svc"].decision_function(grid_6d).reshape(xx.shape)

print(X_2d.shape, Z.shape)


## 11. Scene 2 추론 흐름

핵심 문법:

- frame마다 bbox 내부 포인트 -> feature 추출 -> `svm.predict([feat])`
- GT와 Pred를 각각 다른 색으로 표시
- 프레임별 정확도를 title에 함께 표시

원리:

- 학습이 끝나면 Scene 2의 각 annotation에 대해 객체 종류를 예측한다.
- 이 결과를 3D box 색상으로 보여주면, 수치 평가와 공간적 해석을 동시에 할 수 있다.


In [ ]:
def run_inference(scene_token, frame_idx):
    frame = sync_triplets[scene_token][frame_idx]
    lidar_sd = frame["lidar"]
    pc_full = load_lidar(lidar_sd)
    global2lidar, _ = db.get_sensor_transform(lidar_sd)
    lidar2global = np.linalg.inv(global2lidar)

    pts_h = np.hstack([pc_full[:, :3], np.ones((len(pc_full), 1))])
    pc_global = (lidar2global @ pts_h.T).T[:, :3]

    results = []
    for ann in frame["annotations"]:
        cat = db.get_category(ann)
        if not (cat.startswith("vehicle") or cat.startswith("human.pedestrian")):
            continue
        pts_inside = extract_bbox_points(pc_global, ann)
        if len(pts_inside) < MIN_PTS:
            continue
        feat = compute_features(pts_inside)
        pred = int(svm.predict([feat])[0])
        gt = "vehicle" if cat.startswith("vehicle") else "pedestrian"
        results.append((ann["token"], gt, pred))

    return results

test_scene_token = list(sync_triplets.keys())[1]
results = run_inference(test_scene_token, 0)
print(results[:5])


## 12. 외우면 좋은 핵심 문법

```python
importlib.util.find_spec("plotly")
Path.cwd()
os.chdir(PROJECT_ROOT)
json.load(f)
{r["token"]: r for r in records}
np.fromfile(path, dtype=np.float32).reshape(-1, 5)
np.linalg.inv(T)
np.hstack([pc[:, :3], np.ones((len(pc), 1))])
(pts - center) @ R
np.abs(arr) <= bound
KMeans(n_clusters=K).fit_predict(X)
DBSCAN(eps=0.9, min_samples=20).fit_predict(X)
make_pipeline(StandardScaler(), SVC(...))
accuracy_score(y_true, y_pred)
balanced_accuracy_score(y_true, y_pred)
precision_recall_fscore_support(...)
confusion_matrix(y_true, y_pred)
PCA(n_components=2)
np.meshgrid(...)
pca.inverse_transform(X_2d)
svm.predict([feat])
```
